# Building new membrane compositions  pipline
This notebook contains the pipeline on how the IRE1 systems were created.
Steps of the pipeline:
1. defining membrane compsition, input protein, box size, save location et.c
2. building the system with insane.py
3. correcting topology and index file to match with the input protein (was martinized with CHARMM-GUI)
4. minimization
5. equilibration
6. starting a simulation, monitor CV through out the simulation and stop after initial path between state A and B is achived
7. cut out initial path for the use in AIMMD

In [3]:
import os
import sys
import mdtraj as md
import glob
import numpy as np
sys.path.insert(1, f'{os.getcwd()}/functions')
import functions as functions

# Building the Membrane

## Functions

In [4]:
# function for checking if the simulation is in state A
def stop(subtrajectory:str, stateA:float):
    global frame, index, traj_con
    while True:
        try:
            global index, traj_con
            frame = md.load_frame(subtrajectory, index=index, top=frame)
            drmsd = DRMSD(frame)
            if drmsd < 0.9:
                return True
            traj_con = np.append(traj_con, np.array([[int(index), float(drmsd)]]), axis=0)
            index += 1
        except:

            return False

## Definitions

In [5]:
# systems folder
systems = functions.mkdir(f'../../systems/full_elastic_network')

# Protein
pdb = '../../structures/ire1_martini_2monomer_sep_elastic.pdb'

# Membrane composition (every lipid insane has)
membrane_comp = {'POPC': 100}

# if you need to rename the protein (if not just put: {Protein:1})
new_protein_label = {'PROA': 1,
                     'PROB': 1}

# box dimensions
x = 16
y = 16
z = 11

# salt concentration
salt = 0.15

# getting input for insane.py
name  = '_'.join([f'{key}{membrane_comp[key]}'for key in membrane_comp])
mcomp = ' '.join([f'-l {key}:{membrane_comp[key]}'for key in membrane_comp])

# making input log file
input_log = {'pdb':pdb,
             'membrane composition':name,
             'xyz':f'{x} {y} {z}',
             'salt': 0.15}

# make new directory
di = functions.mkdir(f'{systems}/{name}')
equi_dir = functions.mkdir(f'{di}/equilibration')
min_dir = functions.mkdir(f'{di}/minimization')
initial_dir = functions.mkdir(f'{di}/initial')
functions.copy('sim_params/toppar_cut', f'{di}/toppar', backup=False)

There is already a directory
There is already a directory
There is already a directory
There is already a directory
There is already a directory


## Build membrane with insane

In [4]:
# Do insane.py
insane_input = (f'python2 ../../functions/insane.py -f {pdb} -o {di}/run.gro '
                f'-x {x} -y {y} -z {z} {mcomp} -center -sol W '
                f'-salt {salt} -charge auto -p {di}/topol.top')

functions.trun(insane_input, verbose=False)
input_log['insane input'] = insane_input 
functions.save_log(di, input_log)

## Correct top file

In [5]:
# get lines from the top file
top = [line.strip('\n') for line in open(f'{di}/topol.top', 'r').readlines()
      if line[:8] != '#include']

# get new protein label ready
protein_label = []
for key in new_protein_label:
    protein_label.append(key 
                         + ' '*(len(top[-1])-len(key)-len(str(new_protein_label[key]))) 
                         + str(new_protein_label[key]))

# build new top file with new protein label and correct force field
new_top = ([f'#include "toppar/{file}"' for file in sorted(os.listdir(f'{di}/toppar/'))
         if file[:10] == 'martini_v3']
        + [f'#include "toppar/{file}"' for file in sorted(os.listdir(f'{di}/toppar/'))
           if file[:10] != 'martini_v3']
        + top[:top.index('[ molecules ]')+2]
        + protein_label
        + top[[top.index(line) for line in top if line[:4] == list(membrane_comp.keys())[0]][0]:])

# write new top file
with open(f'{di}/topol.top', 'w') as top_file:
    top_file.truncate()
    for line in new_top:
        top_file.write(line + '\n')
        
input_log['protein name change in topol.top'] = '\n'.join(protein_label)
functions.save_log(di, input_log)

## Create index file

In [6]:
# get input for index file creation
membrane = ' | '.join(str(x) for x in np.arange(13,len(membrane_comp.keys())+13))
solute = len(membrane_comp.keys()) + 16

command = (f'{membrane}\n'
           f'name {len(membrane_comp.keys()) + 17} Membrane\n'
           f'{solute}\n'
           f'name {len(membrane_comp.keys()) + 18} Solute\n'
           f'\n'
           f'q\n')

# make index file
make_ndx = f'gmx_mpi make_ndx -f {di}/run.gro -o {di}/index.ndx -nobackup <<EOF\n{command}EOF'
index_out = functions.trun(make_ndx)

input_log['index file'] = make_ndx
functions.save_log(di, input_log)

# Minimization

In [7]:
# Definitions
min_setup = sorted(glob.glob('sim_params/minimization_setup/*'))
log = []

# copy gro-file into the simulation folder
os.system(f'cp {di}/run.gro {min_dir}/minimized.gro')

for num, step in enumerate(min_setup):
    
    # name
    step_name = step.split('/')[-1][:-4]
    
    # creating tpr file
    grompp = (f'gmx21_mpi grompp -f {step} -c {min_dir}/minimized.gro -p {di}/topol.top'
              f' -o {min_dir}/{step_name}.tpr '
              f' -n {di}/index.ndx -nobackup -maxwarn 1')
    
    functions.trun(grompp)
    
    # doing the minimization step
    mdrun = (f'gmx21_mpi mdrun -ntomp 10 -pin on -deffnm '
             f'{min_dir}/{step_name} -nobackup -v')
    functions.trun(mdrun, open_terminal=True)
    
    log.append('\n'.join([f'step{num} = '+step,
                          'grompp = '+grompp,
                          'mdrun = '+mdrun]))
    os.system(f'cp {min_dir}/{step_name}.gro {min_dir}/minimized.gro')

input_log['minimization steps'] = '\n\n'.join(log)
functions.save_log(di, input_log)

# Equilibration

In [8]:
# Definitions
equi_setup = sorted(glob.glob('sim_params/equibrilation_setup/*'))
log = []

# copy gro-file into the simulation folder
os.system(f'cp {min_dir}/minimized.gro {equi_dir}/equilibrated.gro')

for step in equi_setup:
    
    # name
    step_name = step.split('/')[-1][:-4]
    
    # creating tpr file
    grompp = f'gmx21_mpi grompp -f {step} -c {equi_dir}/equilibrated.gro -p {di}/topol.top'
        f' -o {equi_dir}/{step_name}.tpr -r {equi_dir}/equilibrated.gro'
        f' -n {di}/index.ndx -nobackup -maxwarn 1'
    functions.trun(grompp)
    
    # doing the equilibration step
    mdrun = f'gmx21_mpi mdrun -ntomp 10 -pin on -deffnm '
        f'{equi_dir}/{step_name} -nobackup -v'
    functions.trun(mdrun, open_terminal=True)
    
    log.append('\n'.join([f'step{num} = '+step,
                          'grompp   = '+grompp,
                          'mdrun = '+mdrun]))
    os.system(f'cp {equi_dir}/{step_name}.gro {equi_dir}/equilibrated.gro')

input_log['equilibration steps'] = '\n\n'.join(log)
functions.save_log(di, input_log)

# Getting first transition path

In [27]:
# Definitions
stateA = 0.9
stateB = 3.0
check_every = 60*2

# early setup
functions.copy('sim_params/run.mdp', di)

# run parameters
grompp = (f'gmx21_mpi grompp -f {di}/run.mdp -c {equi_dir}/equilibrated.gro -p {di}/topol.top'
          f' -o {initial_dir}/prun.tpr'
          f' -n {di}/index.ndx -nobackup -maxwarn 1')
mdrun = (f'gmx21_mpi mdrun -ntomp 10 -pin on -deffnm '
         f'{initial_dir}/prun -nobackup -v -dlb yes')
log_tran = '\n'.join(['stateA      = '+str(stateA),
                      'stateB      = '+str(stateB),
                      'check_every = '+str(check_every),
                      'grompp      = '+grompp,
                      'mdrun       = '+mdrun])

In [18]:
# creating tpr file
functions.trun(grompp)
    
# doing the minimization step
initial_out = run(f'{mdrun} -nsteps 1', open_terminal=False)

In [19]:
# prepare
topology = f'{initial_dir}/prun.gro'
frame = md.load(topology)
input_log['generate transition'] = log_tran
functions.save_log(di, input_log)

# main loop
subtrajectory = f'{initial_dir}/prun.xtc'
index = 0
with open(f'{initial_dir}/drmsd_control', "a+", encoding="utf-8") as meta_txt:
    meta_txt.seek(0)
    meta_txt.truncate()
traj_con = np.empty(shape=[0, 2])
try:
    
    while not stop(subtrajectory, stateA=stateA):
        with open(f'{initial_dir}/drmsd_control', "a+", encoding="utf-8") as meta_txt:
            for line in traj_con:
                meta_txt.write(f'{line[0]} {line[1]}\n')
        traj_con = np.empty(shape=[0, 2])
        process = (f'{mdrun} -cpi {initial_dir}/prun.cpt -nsteps -1 -maxh {check_every/3600}')
        cancel(process, verbose=False)
        functions.trun(process, open_terminal=False)
        sleep(1)
except KeyboardInterrupt:
    functions.cancel(process, verbose=True)
    pass

functions.trun(process, open_terminal=False)

## Cut out initial.xtc

In [20]:
# load production run
traj = md.load(f'{initial_dir}/prun.xtc', top=f'{initial_dir}/prun.gro')

In [21]:
# calculate CV form traj
drmsd = functions.DRMSD(traj)

# check states per frame
temp = []
for i in drmsd:
    if i < stateA:
        temp.append(i)
        if np.max(temp) > stateB and np.min(temp) < 0.9:
            break
        temp = [i]
    if i <= stateB and i >= stateA:
        temp.append(i)
    if i > stateB:
        temp.append(i)
        if np.max(temp) > stateB and np.min(temp) < 0.9:
            break
        temp = [i]

# check if there is a transition in the trajectory
if np.max(temp) > stateB and np.min(temp) < 0.9:
    initial_stats = (f'max : {np.round(np.max(temp), 2)} CV\n'
                     f'min : {np.round(np.min(temp), 2)} CV\n'
                     f'time: {len(temp)*traj.timestep*0.001} ns')
    print(initial_stats)
    traj[np.isin(drmsd, temp)].save_xtc(f'{initial_dir}/initial.xtc')
    
    input_log['initial trajectory stats'] = initial_stats
    functions.save_log(di, input_log)
else:
    print(f'No transition in CV range {stateA}-{stateB}')

max : 3.06 CV
min : 0.86 CV
time: 63.0 ns
